In [10]:
import asyncio

# 协程函数：调用后返回 coroutine 对象，不会立刻执行
async def fetch():
    await asyncio.sleep(0.1)  # 模拟 I/O，让出控制权
    return "hello"

async def main():
    # await = "等这个 I/O 完成"
    result = await fetch()
    print(result)

await main()

hello


In [17]:
import asyncio
async def heartbeat():
    for i in range(3):
        print(f"  [heartbeat] tick {i}")
        await asyncio.sleep(0.5)
async def fetch_data():
    print("  [fetch] start")
    await asyncio.sleep(2)
    print("  [fetch] done")
    return {"data": 42}
async def main():
    # 后台启动 heartbeat，不 await —— 立刻继续往下走
    hb_task = asyncio.create_task(heartbeat())
    data = await fetch_data()  # 主流程等 fetch
    print(f"main got: {data}")
    await hb_task  # 收尾：等后台任务结束
await main()

  [fetch] start
  [heartbeat] tick 0
  [heartbeat] tick 1
  [heartbeat] tick 2
  [fetch] done
main got: {'data': 42}


In [18]:
import asyncio
import time
async def call_tool(name: str, delay: float) -> str:
    print(f"  [{name}] start")
    await asyncio.sleep(delay)
    print(f"  [{name}] done")
    return f"{name}_result"
async def main():
    t0 = time.perf_counter()
    # 并发执行 3 个 tool call
    results = await asyncio.gather(
        call_tool("web_search", 1.0),
        call_tool("read_file", 0.5),
        call_tool("run_code", 0.8),
    )
    print(f"results: {results}")
    print(f"elapsed: {time.perf_counter() - t0:.2f}s")  # ~1.0s，不是 2.3s
await main()

  [web_search] start
  [read_file] start
  [run_code] start
  [read_file] done
  [run_code] done
  [web_search] done
results: ['web_search_result', 'read_file_result', 'run_code_result']
elapsed: 1.00s


In [19]:
results = await asyncio.gather(
    call_tool("ok_tool", 0.5),
    call_tool("bad_tool", 0.3),  # 假设会抛异常
    return_exceptions=True,       # 异常变成返回值，不中断其他 task
)
for r in results:
    if isinstance(r, Exception):
        print(f"tool failed: {r}")
    else:
        print(f"tool ok: {r}")

  [ok_tool] start
  [bad_tool] start
  [bad_tool] done
  [ok_tool] done
tool ok: ok_tool_result
tool ok: bad_tool_result


In [ ]:
import asyncio
n = 3
SEMAPHORE = asyncio.Semaphore(n)  # 最多 2 个并发
async def limited_call(agent_id: int):
    async with SEMAPHORE:
        print(f"  agent-{agent_id} running")
        
        await asyncio.sleep(1)

        print(f"  agent-{agent_id} done")
        
async def main():

    asyncio.gather(*[limited_call(i) for i in range(10)])# 10 个 agent，但同时只有 2 个在跑
    print("all agents done")

await main()

all agents done
  agent-0 running
  agent-1 running
  agent-2 running


  agent-0 done
  agent-1 done
  agent-2 done
  agent-3 running
  agent-4 running
  agent-5 running
  agent-3 done
  agent-4 done
  agent-5 done
  agent-6 running
  agent-7 running
  agent-8 running
  agent-6 done
  agent-7 done
  agent-8 done
  agent-9 running
  agent-9 done


In [26]:
import asyncio
from dataclasses import dataclass
@dataclass
class Task:
    id: int
    prompt: str
async def producer(queue: asyncio.Queue, tasks: list[Task]):
    for t in tasks:
        await queue.put(t)
        print(f"[producer] enqueued task-{t.id}")
    # 放 N 个 None 作为 poison pill，通知 worker 退出
    for _ in range(NUM_WORKERS):
        await queue.put(None)
async def worker(worker_id: int, queue: asyncio.Queue, sem: asyncio.Semaphore):
    while True:
        task = await queue.get()
        if task is None:
            queue.task_done()
            break
        async with sem:  # 全局并发上限
            print(f"[worker-{worker_id}] processing task-{task.id}: {task.prompt!r}")
            await asyncio.sleep(0.5)  # 模拟 LLM 调用
            print(f"[worker-{worker_id}] task-{task.id} done")
        queue.task_done()
NUM_WORKERS = 5
async def main():
    queue = asyncio.Queue(maxsize=5)
    sem = asyncio.Semaphore(4)  # 全局最多 2 个 LLM 调用
    tasks = [Task(i, f"prompt-{i}") for i in range(8)]
    await asyncio.gather(
        producer(queue, tasks),
        *[worker(i, queue, sem) for i in range(NUM_WORKERS)],
    )
await (main())

[producer] enqueued task-0
[producer] enqueued task-1
[producer] enqueued task-2
[producer] enqueued task-3
[producer] enqueued task-4
[worker-0] processing task-0: 'prompt-0'
[worker-1] processing task-1: 'prompt-1'
[worker-2] processing task-2: 'prompt-2'
[worker-3] processing task-3: 'prompt-3'
[producer] enqueued task-5
[producer] enqueued task-6
[producer] enqueued task-7
[worker-0] task-0 done
[worker-1] task-1 done
[worker-2] task-2 done
[worker-3] task-3 done
[worker-4] processing task-4: 'prompt-4'
[worker-0] processing task-5: 'prompt-5'
[worker-1] processing task-6: 'prompt-6'
[worker-2] processing task-7: 'prompt-7'
[worker-4] task-4 done
[worker-0] task-5 done
[worker-1] task-6 done
[worker-2] task-7 done


In [1]:
import asyncio
from dataclasses import dataclass, field
from typing import Any
@dataclass
class Message:
    role: str
    content: str
@dataclass
class AgentState:
    messages: list[Message] = field(default_factory=list)
    iteration: int = 0
    max_iterations: int = 5
# ── 模拟 LLM（I/O 密集）──
async def call_llm(messages: list[Message]) -> dict:
    await asyncio.sleep(0.3)
    last = messages[-1].content if messages else ""
    if "search" in last.lower():
        return {"type": "tool_call", "tool": "web_search", "args": {"q": last}}
    return {"type": "text", "content": f"Reply to: {last}"}
# ── 模拟 Tool（I/O 密集）──
async def execute_tool(name: str, args: dict) -> str:
    await asyncio.sleep(0.5)
    return f"[{name}]({args}) => mock result"
# ── 核心：Turn Scheduler（≈ conversation_loop）──
async def run_turn(state: AgentState, user_msg: str) -> str:
    state.messages.append(Message("user", user_msg))
    state.iteration = 0
    while state.iteration < state.max_iterations:
        state.iteration += 1
        print(f"\n── iteration {state.iteration} ──")
        # Phase 1: LLM call（await 让出控制权）
        response = await call_llm(state.messages)
        if response["type"] == "text":
            state.messages.append(Message("assistant", response["content"]))
            return response["content"]
        # Phase 2: Tool dispatch（可并行）
        if response["type"] == "tool_call":
            tool_result = await execute_tool(response["tool"], response["args"])
            state.messages.append(Message("tool", tool_result))
            # loop 继续 → 把 tool result 喂回 LLM
    return "[halted] max iterations reached"
async def main():
    state = AgentState()
    answer = await run_turn(state, " python search ")
    print(f"\nFinal: {answer}")
    print(f"Total iterations: {state.iteration}")
await main()


── iteration 1 ──

── iteration 2 ──

── iteration 3 ──

── iteration 4 ──

── iteration 5 ──

Final: [halted] max iterations reached
Total iterations: 5


In [ ]:
import asyncio
import threading
import concurrent.futures
# 进程级持久 loop（CLI 路径用）
_tool_loop: asyncio.AbstractEventLoop | None = None
_worker_local = threading.local()
def get_tool_loop() -> asyncio.AbstractEventLoop:
    global _tool_loop
    if _tool_loop is None or _tool_loop.is_closed():
        _tool_loop = asyncio.new_event_loop()
    return _tool_loop
def get_worker_loop() -> asyncio.AbstractEventLoop:
    loop = getattr(_worker_local, "loop", None)
    if loop is None or loop.is_closed():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        _worker_local.loop = loop
    return loop
def run_async(coro):
  """简化版 Hermes _run_async:sync 上下文里跑 async coroutine"""
    try:
        running = asyncio.get_running_loop()
    except RuntimeError:
        running = None
    if running and running.is_running():
        # 已在 async 上下文（gateway）→ 开新线程 + 新 loop
        def _worker():
            loop = asyncio.new_event_loop()
            try:
                return loop.run_until_complete(coro)
            finally:
                loop.close()
        with concurrent.futures.ThreadPoolExecutor(1) as pool:
            return pool.submit(_worker).result(timeout=30)
    if threading.current_thread() is not threading.main_thread():
        # worker thread → 每线程一个持久 loop
        return get_worker_loop().run_until_complete(coro)
    # 主线程 CLI → 进程级持久 loop
    return get_tool_loop().run_until_complete(coro)
    # ── 使用示例 ──
async def async_tool(query: str) -> str:
    await asyncio.sleep(0.2)
    return f"result for {query}"
# sync 调用方（像 Hermes 的 handle_function_call）
def handle_function_call(name: str, args: dict) -> str:
    if name == "web_search":
        return run_async(async_tool(args["q"]))
    raise ValueError(name)
print(handle_function_call("web_search", {"q": "hermes agent"}))


IndentationError: unexpected indent (1026099349.py, line 21)